# NYC Taxi Data Preparation

## 1. Question and provenance

**All input rows are synthetic. Reconstructed public demo; not a reproduction of the historical sample.**

How do source identity, tip policies, time normalization, joins and sampling change the records available for analysis? This walkthrough calls the repository's preparation functions and CLI API. Historical 780M-row and model-performance claims are outside its scope.

Recovered rules and source digests are recorded in [the preparation report](../reports/data_preparation.md).

In [1]:
from pathlib import Path
import copy
import json
import sys
import tempfile
import pandas as pd
from IPython.display import display

cwd = Path.cwd().resolve()
ROOT = next((p for p in (cwd, cwd.parent)
             if (p / 'configs/data_prep_demo.json').is_file()
             and (p / 'scripts/build_trip_sample.py').is_file()), None)
if ROOT is None:
    raise RuntimeError('Start in the repository root or notebooks directory.')
sys.path.insert(0, str(ROOT / 'scripts'))
from prepare_trips import load_policy, read_trip_source, normalize_trips, filter_trips
from prep_joins import join_zones, join_weather
from prep_sampling import assign_splits, sample_trips
from build_trip_sample import run_preparation

CONFIG = ROOT / 'configs/data_prep_demo.json'
INPUTS = ROOT / 'tests/fixtures/data_prep'
policy = load_policy(CONFIG)
raw, source = read_trip_source(INPUTS / 'trips.csv', 'synthetic_trips')
print(f'Synthetic source: {len(raw)} records; policy: {policy["status"]}')

Synthetic source: 33 records; policy: reconstructed_public_demo


## 2. Raw schema and units

Raw fields remain unchanged; normalized copies explicitly use USD, miles, minutes and a fractional tip rate. Original record ordinals are attached before filtering. `trip_id` identifies a physical record in a byte-identical file; a separate fingerprint flags matching values without deleting them.

In [2]:
display(pd.DataFrame([
    ['fare_amount / tip_amount', 'USD', 'Original numeric representation retained'],
    ['trip_distance', 'miles', 'Missing is not zero'],
    ['pickup / dropoff', 'New York local', 'Naive input; DST cases quarantined'],
    ['weather observed / available', 'UTC offsets', 'Observation and release time are separate'],
    ['tip_rate_fare', 'fraction', '0.2 means 20%, not 0.2%'],
], columns=['Field', 'Unit', 'Meaning']))
display(raw[['fixture_case', 'fare_amount', 'tip_amount', 'trip_distance', 'payment_type']].head(5))

,Field,Unit,Meaning
0,fare_amount / tip_amount,USD,Original numeric representation retained
1,trip_distance,miles,Missing is not zero
2,pickup / dropoff,New York local,Naive input; DST cases quarantined
3,weather observed / available,UTC offsets,Observation and release time are separate
4,tip_rate_fare,fraction,"0.2 means 20%, not 0.2%"


,fixture_case,fare_amount,tip_amount,trip_distance,payment_type
0,normal,10,2,2,1
1,identical_content,10,2,2,1
2,zero_tip,10,0,2,1
3,refund,10,-2,2,1
4,zero_fare,0,2,2,1


## 3. Cleaning and target definitions

| Decision | Source | Alternative | Consequence |
| --- | --- | --- | --- |
| Keep zero-tip card records | Recovered weather sample policy | Positive-tip subset | Changes the estimand |
| Reject invalid denominators/refunds | Recovered logic; demo finite checks | Separate refund analysis | Exclusions remain auditable |
| Quarantine DST gap/fold | Demo assumption | Explicit offset from source | No guessed UTC time |
| No upper caps by default | Demo assumption | Recovered historical caps | Demo is not the historical sample |

In [3]:
normalized, time_ledger = normalize_trips(raw, source, policy)
all_card, economic_ledger = filter_trips(normalized, policy)
positive_policy = copy.deepcopy(policy)
positive_policy['filter']['sample_policy'] = 'positive_tip'
positive, _ = filter_trips(normalized, positive_policy)
comparison = normalized.loc[normalized.source_row_number.isin([0, 1, 2, 8]),
                            ['fixture_case', 'tip_usd', 'tip_or_not', 'tip_rate_fare']].copy()
comparison['all_card'] = comparison.index.isin(all_card.index)
comparison['positive_tip'] = comparison.index.isin(positive.index)
display(comparison.reset_index(drop=True))
print(f'All nonnegative cards: {len(all_card)}; positive-tip cards: {len(positive)}')

,fixture_case,tip_usd,tip_or_not,tip_rate_fare,all_card,positive_tip
0,normal,2,1,0.2,True,True
1,identical_content,2,1,0.2,True,True
2,zero_tip,0,0,0.0,True,False
3,cash_recording,0,<NA>,NaN,False,False


All nonnegative cards: 23; positive-tip cards: 16


In [4]:
display(time_ledger[['source_row_number', 'reason', 'is_primary']])
reason_counts = economic_ledger.groupby('reason', as_index=False).agg(
    overlapping_rows=('trip_id', 'size'), primary_rows=('is_primary', 'sum'))
display(reason_counts)
assert len(raw) == len(normalized) + time_ledger.loc[time_ledger.is_primary].trip_id.nunique()
assert len(normalized) == len(all_card) + economic_ledger.loc[economic_ledger.is_primary].trip_id.nunique()

,source_row_number,reason,is_primary
0,14,pickup_time_invalid,True
1,12,pickup_time_ambiguous,True
2,11,pickup_time_nonexistent,True


,reason,overlapping_rows,primary_rows
0,distance_not_positive_finite,2,1
1,duration_not_positive_finite,1,1
2,fare_not_positive_finite,3,3
3,non_card_or_unknown_payment,1,1
4,tip_negative_or_nonfinite,2,1


## 4. Zone and weather joins

Zone keys must be unique. Both pickup and dropoff use many-to-one left joins.
Weather uses the explicitly mapped synthetic station and a 60-minute nearest-observation tolerance. A tie picks the earlier observation. This is descriptive: later observations are allowed and flagged. `asof_available` is deliberately refused.

In [5]:
zones = pd.read_csv(INPUTS / 'zones.csv')
weather = pd.read_csv(INPUTS / 'weather.csv')
zoned, zone_audit = join_zones(all_card, zones)
joined, weather_audit = join_weather(zoned, weather, policy)
display(joined.loc[joined.source_row_number.isin([0, 9, 10]),
    ['fixture_case', 'pickup_zone_matched', 'dropoff_zone_matched',
     'weather_matched', 'weather_lag_minutes']].reset_index(drop=True))
future = all_card.iloc[:1].copy()
future['pickup_utc'] = pd.to_datetime(['2024-01-15T15:50:00Z'])
future_join, _ = join_weather(future, weather, policy)
display(future_join[['weather_record_id', 'weather_lag_minutes',
                    'weather_observation_after_pickup', 'weather_available_after_pickup']])
assert joined.trip_id.tolist() == all_card.trip_id.tolist()

,fixture_case,pickup_zone_matched,dropoff_zone_matched,weather_matched,weather_lag_minutes
0,normal,True,True,True,-30.0
1,unknown_pickup_zone,False,True,False,NaN
2,weather_gap,True,False,False,NaN


,weather_record_id,weather_lag_minutes,weather_observation_after_pickup,weather_available_after_pickup
0,SYN_W2,10.0,True,True


In [6]:
try:
    join_zones(all_card, pd.concat([zones, zones.iloc[:1]], ignore_index=True))
except ValueError:
    print('Expected rejection: duplicate Zone keys would multiply trip rows.')
else:
    raise AssertionError('Duplicate Zone key was not rejected.')

Expected rejection: duplicate Zone keys would multiply trip rows.


## 5. Filtering flow and saved artifacts

Run the same pipeline exposed by the CLI in a new temporary directory under ignored `.runs/`. The demonstration checks row conservation, joins and exclusion counts. No input, historical result or previous run is overwritten.

In [7]:
(ROOT / '.runs').mkdir(exist_ok=True)
with tempfile.TemporaryDirectory(prefix='notebook-', dir=ROOT / '.runs') as directory:
    output = Path(directory)
    quality = run_preparation(CONFIG, INPUTS, output)
    flow = pd.read_csv(output / 'sample_flow.csv')
    saved_sample = pd.read_parquet(output / 'trip_sample.parquet')
    sampling_audit = pd.read_csv(output / 'sampling_audit.csv')
    manifest = json.loads((output / 'source_manifest.json').read_text())
display(flow[['stage', 'input_rows', 'output_rows', 'removed_rows']])
assert (flow.input_rows == flow.output_rows + flow.removed_rows).all()
print('Source statuses:', sorted({entry['status'] for entry in manifest['sources']}))
print('Selected ID digest:', quality['sample_id_sha256'])

,stage,input_rows,output_rows,removed_rows
0,source_records,33,33,0
1,normalize,33,30,3
2,filter,30,23,7
3,zone_join,23,23,0
4,weather_join,23,23,0
5,match_filter,23,23,0
6,split,23,23,0
7,sample,23,14,9


Source statuses: ['synthetic']
Selected ID digest: 9a55be5a9c916d078107e209ee6dd1344abe1aed56bf28d1271f264ce4837cef


## 6. Stable sampling and splits

SHA256 assigns retained content-duplicate groups to one of three within-period splits. Sampling then ranks stable trip IDs with the configured seed and takes a fixed cap per split and tip stratum. This is not a future-period forecasting test. Existing valid buckets are preserved; cross-split duplicate-group conflicts are rejected. No learned transformation is fitted here.

In [8]:
split = assign_splits(joined, policy)
sample_a, audit_a = sample_trips(split, policy)
shuffled = joined.sample(frac=1, random_state=17)
sample_b, audit_b = sample_trips(assign_splits(shuffled, policy), policy)
pd.testing.assert_frame_equal(sample_a, sample_b)
pd.testing.assert_frame_equal(audit_a, audit_b)
assert split.groupby('record_fingerprint').split_label.nunique().max() == 1
assert sample_a.trip_id.nunique() == len(sample_a)
display(sampling_audit[['split_label', 'stratum', 'eligible_n', 'sample_n']])
print('Input order invariance verified; suspected duplicate groups do not cross splits.')

,split_label,stratum,eligible_n,sample_n
0,train,0,6,4
1,train,1,11,4
2,validation,0,1,1
3,validation,1,3,3
4,test,0,0,0
5,test,1,2,2


Input order invariance verified; suspected duplicate groups do not cross splits.


## 7. What this demonstration establishes

The synthetic run verifies explicit source identity, row accounting, target-policy differences, DST handling, join cardinality and deterministic sampling. It does **not** reproduce the historical 132-file / 780M-row / 89.6M-card-trip workflow or its R-squared values, and it does not establish weather causality.

The output lacks several fare controls and richer weather fields required by the historical weather models. ACS/PLUTO/MTA builders, spatial panels, Knicks event tagging, large-scale throughput and the old OLS/LightGBM same-sample comparison remain outside this change. See the [consumer boundary and provenance report](../reports/data_preparation.md).